In [ ]:
%pip install torch torchvision pandas torchmetrics timm wakepy onnxruntime

In [6]:
import os 
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
import pandas as pd
from model import Model
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy, F1Score, Recall, Precision
from torchmetrics import MetricCollection
from wakepy import keep


# 1. 하이퍼파라미터 및 디바이스 설정

In [15]:
BATCH_SIZE = 64
EPOCHS = 300
LR = 1e-3
IMGZ = (384, 384)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 데이터 전처리 및 로더 설정

In [16]:
train_transform = transforms.Compose([
    transforms.Resize(IMGZ),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(hue=0.015, saturation=0.7, brightness=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomErasing(p=0.4)
])

val_transform = transforms.Compose([
    transforms.Resize(IMGZ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

image_datasets = {
    'train': datasets.ImageFolder('./dataset/train', transform=train_transform),
    'val': datasets.ImageFolder('./dataset/val', transform=val_transform)
}

dataloaders = {
    phase: DataLoader(image_datasets[phase], batch_size=BATCH_SIZE, shuffle=(phase == 'train'))
    for phase in ['train', 'val']
}

# 3. 모델, 손실함수, 옵티마이저 & 저장 경로

In [17]:
model = Model(image_datasets['train'].classes, 'tf_efficientnetv2_s.in21k_ft_in1k').to(DEVICE)
criterion = CrossEntropyLoss()
optimizer = AdamW(model.classifier.parameters(), lr=LR)

folder_index = 0
while os.path.exists(save_path := f"run/train{'' if folder_index == 0 else folder_index}"):
    folder_index += 1
os.makedirs(save_path)
print(f"저장 경로: {save_path}")

with open(f"{save_path}/classes.txt", "w") as file:
    file.write("\n".join(model.classes))

저장 경로: run/train4


# 4. 평가지표 설정

In [18]:
base_metrics = MetricCollection({
    'Acc': Accuracy(task='multiclass', num_classes=model.num_classes),
    'F1': F1Score(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Prec': Precision(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Rec': Recall(task='multiclass', num_classes=model.num_classes, average='macro')
})

metrics = {
    phase: base_metrics.clone(prefix=f'{phase}_').to(DEVICE)
    for phase in ['train', 'val']
}

# 5. 학습 및 검증 루프 (wakepy 화면 켜짐 유지)

In [19]:
best_val_acc = 0.0
history = []
best_val_loss=float('inf')
dump=(torch.randn(1,3,IMGZ[0],IMGZ[1]),)

with keep.presenting():
    for epoch in range(EPOCHS):
        epoch_results = {}
        
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            metrics[phase].reset()
            
            with torch.set_grad_enabled(phase == 'train'):
                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    
                    if phase == 'train':optimizer.zero_grad()
                        
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        
                    running_loss += loss.item() * inputs.size(0)
                    metrics[phase].update(outputs, labels)
                    
            phase_loss = running_loss / len(image_datasets[phase])
            phase_metrics = {name: val.item() for name, val in metrics[phase].compute().items()}
            
            epoch_results.update(phase_metrics)
            epoch_results[f'{phase}_Loss'] = phase_loss
            
            metrics_str = " | ".join(f"{name}: {val:.4f}" for name, val in phase_metrics.items())
            print(f"Epoch {epoch+1}/{EPOCHS} [{phase.upper()}] Loss: {phase_loss:.4f} | {metrics_str}")
            
        history.append(epoch_results)
        
        # CSV 저장 시 에폭 인덱스를 1부터 시작
        history_df = pd.DataFrame(history)
        history_df.index = history_df.index + 1
        history_df.to_csv(f'{save_path}/result.csv', index_label="epoch")
        onnx_model=torch.onnx.export(model,dump,dynamo=True)
        onnx_model.save(f"{save_path}/last.onnx")
        val_acc, val_loss = epoch_results['val_Acc'], epoch_results['val_Loss']
        if (val_acc, -val_loss) > (best_val_acc, -best_val_loss):
            best_val_acc, best_val_loss = val_acc, val_loss
            onnx_model.save(f"{save_path}/best.onnx")

Epoch 1/300 [TRAIN] Loss: 6.4835 | train_Acc: 0.2668 | train_F1: 0.2622 | train_Prec: 0.2672 | train_Rec: 0.2642
Epoch 1/300 [VAL] Loss: 6.8122 | val_Acc: 0.2718 | val_F1: 0.2317 | val_Prec: 0.2928 | val_Rec: 0.2700
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Epoch 2/300 [TRAIN] Loss: 5.3177 | train_Acc: 0.2941 | train_F1: 0.2923 | train_Prec: 0.2930 | train_Rec: 0.2919
Epoch 2/300 [VAL] Loss: 6.5968 | val_Acc: 0.3051 | val_F1: 0.2626 | val_Prec: 0.3352 | val_Rec: 0.2974
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Epoch 3/300 [TRAIN] Loss: 4.8437 | train_Acc: 0.3129 | train_F1: 0.3108 | train_Prec: 0.3122 | train_Rec: 0.3105
Epoch 3/300 [VAL] Loss: 6.4825 | val_Acc: 0.3051 | val_F1: 0.2545 | val_Prec: 0.3339 | val_Rec: 0.2980
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


...

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Epoch 202/300 [TRAIN] Loss: 0.9056 | train_Acc: 0.6484 | train_F1: 0.6459 | train_Prec: 0.6457 | train_Rec: 0.6466
Epoch 202/300 [VAL] Loss: 1.6704 | val_Acc: 0.5179 | val_F1: 0.4932 | val_Prec: 0.5710 | val_Rec: 0.5120
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Model([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


KeyboardInterrupt: 